# 8. Hypotesprövning och p-värden

Ett konfidensintervall svarar på frågan *"vad är värdet, ungefär?"*. En **hypotesprövning** svarar på en ja/nej-fråga:
*"är det här resultatet mer än vad slumpen kan förklara?"*

Tankesättet är lite bakvänt första gången, så vi börjar med ett mynt och bygger p-värdet från grunden genom simulering.
Formlerna kommer efteråt.

**Efter den här notebooken kan du**
- formulera nollhypotes och alternativhypotes
- förklara vad ett p-värde är, och vad det *inte* är
- genomföra och tolka ett t-test för ett medelvärde och för skillnaden mellan två medelvärden
- skilja på ensidig och dubbelsidig prövning
- se sambandet mellan hypotesprövning och konfidensintervall

**Förkunskaper:** [7. Punktskattning och konfidensintervall](../07_konfidensintervall/07_konfidensintervall.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

N = 10_000
population = pd.DataFrame({
    "height": rng.normal(172, 9, N).round(1),                       # cm
    "num_children": rng.poisson(1.5, N),
    "education": rng.choice(["Grundskola", "Gymnasium", "Högskola"], N, p=[0.2, 0.45, 0.35]),
    "income": rng.lognormal(np.log(32), 0.4, N).round(1),           # tkr per månad
})
mu = population["height"].mean()                                 # sanningen – okänd i verkligheten
height = population.sample(n=100, random_state=1)["height"]      # vårt stickprov

## 8.1 Är myntet rättvist?

En vän påstår att hens mynt är helt vanligt. Du singlar det 100 gånger och får **62 krona**.
Ett rättvist mynt borde ge ungefär 50. Är 62 tillräckligt konstigt för att du ska misstänka fusk?

Frågan går inte att svara på utan att veta **hur mycket ett rättvist mynt brukar variera**. Så det tar vi reda på:
vi låter datorn singla ett garanterat rättvist mynt 100 gånger, och upprepar det 10 000 gånger.

In [ ]:
observed_heads = 62        # ← ändra: pröva 55, 58, 62 och 70

fair_results = rng.binomial(100, 0.5, 10_000)      # antal krona i 10 000 experiment med rättvist mynt

distance = abs(observed_heads - 50)
as_extreme = np.abs(fair_results - 50) >= distance

plt.hist(fair_results, bins=np.arange(24.5, 76.5, 1), color="lightgray", edgecolor="white", label="rättvist mynt")
plt.hist(fair_results[as_extreme], bins=np.arange(24.5, 76.5, 1), color="red", edgecolor="white",
         label="minst lika extremt som vårt resultat")
plt.axvline(observed_heads, color="black", linestyle="--", label=f"observerat: {observed_heads}")
plt.xlabel("Antal krona av 100")
plt.ylabel("Antal experiment")
plt.legend()
plt.show()

print(f"Andel experiment minst {distance} från 50: {as_extreme.mean():.3f}")

Den röda andelen är ett **p-värde**. Det svarar på frågan:

> *Om myntet är rättvist, hur ofta får man då ett resultat som är minst så extremt som det vi såg?*

Med 62 krona är svaret ungefär 2 %. Ett rättvist mynt *kan* ge 62 krona, men det är ovanligt. Då står vi inför två förklaringar:
antingen hade vi otur, eller så är myntet inte rättvist. Ju mindre p-värdet är, desto svårare blir det att tro på oturen.

🧪 **Labba:** Pröva 55, 58 och 70 krona. Vid vilket antal börjar *du* tvivla på myntet?

Lägg märke till att vi räknar avvikelser åt **båda håll** (38 krona eller färre är lika konstigt som 62 eller fler).
Det kallas en dubbelsidig prövning, och vi återkommer till det.

Simuleringen kan ersättas med en exakt beräkning:

In [ ]:
print(f"Exakt p-värde: {stats.binomtest(62, 100, 0.5).pvalue:.3f}")

## 8.2 Arbetsgången

Det du just gjorde är en hypotesprövning. Formellt ser den ut så här:

| Steg | | Myntexemplet |
|---|---|---|
| 1 | Formulera **nollhypotes** $H_0$ (ingen effekt, ingen skillnad) och **alternativhypotes** $H_1$ | $H_0$: myntet är rättvist (p = 0,5). $H_1$: p ≠ 0,5 |
| 2 | Välj **signifikansnivå** α, oftast 0,05 | α = 0,05 |
| 3 | Beräkna en **testvariabel** från datan | 62 krona |
| 4 | Beräkna **p-värdet**: sannolikheten för ett minst så extremt resultat, *om $H_0$ är sann* | p ≈ 0,02 |
| 5 | **Slutsats:** om p < α förkastas $H_0$ | 0,02 < 0,05: förkasta $H_0$ |

När $H_0$ förkastas säger man att resultatet är **statistiskt säkerställt** eller **signifikant**.

Logiken liknar en rättegång: den åtalade ($H_0$) antas vara oskyldig tills bevisen (datan) är tillräckligt starka.
Signifikansnivån α är gränsen för "tillräckligt starka", och den ska bestämmas *innan* man tittar på datan.

### Två vanliga missförstånd

> **1. Att inte förkasta $H_0$ betyder inte att $H_0$ är bevisad.** Med 55 krona av 100 kan vi inte förkasta att myntet
> är rättvist, men det kan mycket väl vara lite snett ändå. "Frikänd i brist på bevis" är inte samma sak som "bevisat oskyldig".
>
> **2. p-värdet är inte sannolikheten att $H_0$ är sann.** Det räknas ut *under antagandet* att $H_0$ är sann
> och anger hur ovanlig vår data då vore.

## 8.3 t-test för ett medelvärde

Nu samma logik för ett medelvärde. Någon påstår att medellängden i populationen är **170 cm**.
Vi har ett stickprov med 100 personer.

- $H_0: \mu = 170$
- $H_1: \mu \neq 170$

In [ ]:
n = len(height)
x_bar = height.mean()
std_error = height.std() / np.sqrt(n)
print(f"n = {n},  x̄ = {x_bar:.2f} cm,  medelfel = {std_error:.2f} cm")

Stickprovets medelvärde ligger en bit över 170. Men som med myntet är frågan om avvikelsen är stor **jämfört med vad
slumpen brukar ge**, och slumpens storlek är just medelfelet. Testvariabeln blir därför

$$t = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}$$

alltså *hur många medelfel* stickprovsmedelvärdet ligger från det påstådda värdet. Det är ett z-värde från
[notebook 4](../04_fordelningar/04_fordelningar.ipynb), fast för ett medelvärde. Om $H_0$ är sann följer t en t-fördelning
med n − 1 frihetsgrader.

In [ ]:
mu_0 = 170          # ← ändra: pröva 168, 169, 171, 172 och 175

t_obs = (x_bar - mu_0) / std_error
p_value = 2 * stats.t.sf(abs(t_obs), df=n - 1)       # arean i båda svansarna

print(f"H0: μ = {mu_0}")
print(f"Testvariabel t = {t_obs:.2f}")
print(f"p-värde        = {p_value:.4f}")

x = np.linspace(-5, 5, 400)
y = stats.t.pdf(x, df=n - 1)
plt.plot(x, y, color="black")
plt.fill_between(x, y, where=np.abs(x) >= abs(t_obs), color="red", alpha=0.4, label=f"p-värde = {p_value:.4f}")
plt.axvline(t_obs, color="red", linestyle="--", label=f"t = {t_obs:.2f}")
plt.xlabel("t")
plt.ylabel("Täthet")
plt.title("Testvariabelns fördelning om H0 är sann")
plt.legend()
plt.show()

p-värdet är den röda arean: sannolikheten att hamna minst så långt ut i svansarna om $H_0$ är sann.
Det är samma röda staplar som i myntexemplet, fast med en kurva i stället för ett histogram.

🧪 **Labba:** Pröva olika värden på `mu_0`.
- För vilka värden blir p < 0,05?
- Vad händer med t och p när `mu_0` närmar sig x̄?

scipy gör samma test på en rad:

In [ ]:
mu_0 = 170
res = stats.ttest_1samp(height, mu_0)
print(f"t = {res.statistic:.2f},  p-värde = {res.pvalue:.4f}")

Ett test är inte klart förrän slutsatsen är formulerad **i ord**:

In [ ]:
alpha = 0.05
if res.pvalue < alpha:
    print(f"p = {res.pvalue:.4f} < {alpha}: förkasta H0. Medellängden skiljer sig signifikant från {mu_0} cm.")
else:
    print(f"p = {res.pvalue:.4f} ≥ {alpha}: H0 kan inte förkastas. Vi saknar belägg för att medellängden skiljer sig från {mu_0} cm.")

> **Fundera:** Vi har facit och vet att det sanna μ är ungefär 171,9 cm. $H_0: \mu = 170$ är alltså **falsk**, men testet
> lyckades inte upptäcka det. Det kallas ett **typ II-fel**, och det är precis missförstånd 1 ovan i praktiken.
> Mer om detta i [notebook 10](../10_typ1_typ2_fel/10_typ1_typ2_fel.ipynb).

### Sambandet med konfidensintervallet

✏️ **Din tur:** Räkna ut 95 % konfidensintervallet för medellängden (du kan använda
`stats.t.interval(0.95, df=n - 1, loc=x_bar, scale=std_error)`). Jämför med det du såg i labben ovan:
vilka värden på `mu_0` förkastades, och vilka gjorde det inte? Ser du mönstret?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
low, high = stats.t.interval(0.95, df=n - 1, loc=x_bar, scale=std_error)
print(f"95 % KI: [{low:.2f}, {high:.2f}]")
for candidate in [168, 169, 170, 171, 172, 173, 175]:
    p_val = stats.ttest_1samp(height, candidate).pvalue
    inside = low <= candidate <= high
    print(f"μ0 = {candidate}: p = {p_val:.4f}   i intervallet: {inside}   förkastas: {p_val < 0.05}")
```

</details>

<details>
<summary>💡 <b>Mönstret</b></summary>

Värden **inuti** 95 %-intervallet förkastas inte på nivån 5 %. Värden **utanför** förkastas. Konfidensintervallet är alltså
mängden av alla nollhypoteser som datan inte motsäger. Därför ger intervallet mer information än ett ensamt p-värde:
det visar alla rimliga värden på en gång.

</details>

## 8.4 Ensidig och dubbelsidig prövning

- **Dubbelsidig:** $H_1: \mu \neq 170$. Avvikelser åt båda håll räknas.
- **Ensidig:** $H_1: \mu > 170$ (eller $<$). Bara en riktning räknas.

Ensidig prövning används när bara den ena riktningen är intressant, till exempel "ger den nya metoden *bättre* resultat?".
Riktningen måste bestämmas **innan** man tittar på datan.

🔮 **Gissa först:** x̄ ligger över 170. Vilket av de tre p-värdena blir minst, och vilket blir störst?

In [ ]:
for alternative, label in [("two-sided", "H1: μ ≠ 170"), ("greater", "H1: μ > 170"), ("less", "H1: μ < 170")]:
    p_val = stats.ttest_1samp(height, 170, alternative=alternative).pvalue
    print(f"{label}:  p = {p_val:.4f}")

Det ensidiga p-värdet är hälften av det dubbelsidiga **när avvikelsen går åt det håll som $H_1$ pekar ut**.
För $H_1: \mu < 170$ pekar datan åt fel håll, och p-värdet blir stort.

Att välja riktning efter att ha sett datan är ett sätt att i smyg halvera p-värdet. Är du osäker: välj dubbelsidigt.

## 8.5 t-test för skillnaden mellan två medelvärden

Den vanligaste frågan i praktiken är inte "är medelvärdet 170?" utan **"skiljer sig två grupper åt?"**

Två undervisningsmetoder jämförs. Eleverna lottas till metod A eller B och skriver sedan samma prov.

- $H_0: \mu_A = \mu_B$ (ingen skillnad i medelpoäng)
- $H_1: \mu_A \neq \mu_B$

In [ ]:
group_size = 40         # ← ändra: pröva 10 och 200
true_mean_a = 65        # sanningen i simuleringen
true_mean_b = 74        # ← ändra: pröva 65 (ingen verklig skillnad) och 67 (liten skillnad)

rng_exp = np.random.default_rng(2)       # eget frö, så att cellen ger samma data varje gång den körs
method_a = rng_exp.normal(true_mean_a, 12, group_size).round()
method_b = rng_exp.normal(true_mean_b, 12, group_size).round()

print(f"Medelpoäng A: {method_a.mean():.1f}   B: {method_b.mean():.1f}   Skillnad (A − B): {method_a.mean() - method_b.mean():.1f}")

**Titta alltid på datan innan du testar:**

In [ ]:
plt.boxplot([method_a, method_b])
plt.xticks([1, 2], ["Metod A", "Metod B"])
plt.ylabel("Poäng")
plt.title("Provresultat per metod")
plt.show()

Grupperna överlappar. Är skillnaden i medelvärde större än vad slumpen kan förklara? Testvariabeln följer samma mönster som förut:

$$t = \frac{\text{observerad skillnad}}{\text{skillnadens medelfel}} = \frac{\bar{x}_A - \bar{x}_B}{\sqrt{s_A^2/n_A + s_B^2/n_B}}$$

In [ ]:
diff = method_a.mean() - method_b.mean()
std_error_diff = np.sqrt(method_a.var(ddof=1) / len(method_a) + method_b.var(ddof=1) / len(method_b))
print(f"Skillnad: {diff:.2f}   medelfel: {std_error_diff:.2f}   t = {diff / std_error_diff:.2f}")

In [ ]:
res = stats.ttest_ind(method_a, method_b, equal_var=False)       # Welchs t-test
ci = res.confidence_interval(0.95)

print(f"t = {res.statistic:.2f},  p-värde = {res.pvalue:.4f}")
print(f"95 % KI för skillnaden (A − B): [{ci.low:.1f}, {ci.high:.1f}] poäng")
if res.pvalue < alpha:
    print("Slutsats: förkasta H0. Skillnaden i medelpoäng är statistiskt säkerställd.")
else:
    print("Slutsats: H0 kan inte förkastas. Vi saknar belägg för en skillnad i medelpoäng.")

- **`equal_var=False`** ger Welchs t-test, som inte förutsätter att grupperna har samma spridning. Använd det som standard.
- **Tecknet på t** visar bara riktningen. scipy räknar A − B, så t blir negativt när A har lägre medelpoäng.
- **Konfidensintervallet** visar *hur stor* skillnaden kan vara, vilket p-värdet inte gör. Ett intervall som inte innehåller 0
  säger samma sak som p < 0,05.
- **Förutsättningar:** två oberoende slumpmässiga stickprov, och ungefär normalfördelade värden eller tillräckligt stora grupper.

🧪 **Labba:** Gå tillbaka till cellen där datan skapas, ändra en sak i taget och kör om cellerna därifrån och hit.
1. `group_size = 10`. Den sanna skillnaden är fortfarande 9 poäng. Hur brett blir konfidensintervallet? Upptäcker testet skillnaden?
2. `group_size = 40`, `true_mean_b = 65`. Nu är $H_0$ sann. Vad säger testet?
3. `true_mean_b = 67`. En liten verklig skillnad. Pröva `group_size` 40 och sedan 500.

<details>
<summary>💡 <b>Vad bör du ha sett?</b></summary>

1. Med 10 per grupp blir intervallet mycket brett och innehåller 0. Skillnaden blir icke-signifikant trots att den finns:
   testet har för lite data för att skilja den från slumpen.
2. Ett stort p-värde, och ett intervall som innehåller 0.
3. Med 40 per grupp upptäcks inte den lilla skillnaden. Med 500 per grupp blir även 2 poängs skillnad tydligt signifikant.

Om ett test blir signifikant beror alltså på **tre saker**: hur stor skillnaden är, hur stor spridningen är och hur mycket data man har.

</details>

✏️ **Din tur:** Ett företag testar om en ny rutin kortar handläggningstiden. Tider i minuter:

```python
old_routine = np.array([34, 41, 38, 45, 36, 40, 43, 39, 37, 42, 44, 35])
new_routine = np.array([31, 36, 33, 39, 30, 35, 38, 34, 37, 32, 36, 33])
```

1. Formulera $H_0$ och $H_1$. Företaget bryr sig bara om ifall den nya rutinen är *snabbare*: ensidigt eller dubbelsidigt?
2. Rita ett lådagram.
3. Gör testet och formulera slutsatsen i ord.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
old_routine = np.array([34, 41, 38, 45, 36, 40, 43, 39, 37, 42, 44, 35])
new_routine = np.array([31, 36, 33, 39, 30, 35, 38, 34, 37, 32, 36, 33])

# H0: μ_gammal = μ_ny.  H1: μ_gammal > μ_ny (ensidigt, bestämt i förväg).
plt.boxplot([old_routine, new_routine])
plt.xticks([1, 2], ["Gammal rutin", "Ny rutin"])
plt.ylabel("Minuter")
plt.show()

res_routine = stats.ttest_ind(old_routine, new_routine, equal_var=False, alternative="greater")
print(f"Medel gammal: {old_routine.mean():.1f}   ny: {new_routine.mean():.1f}")
print(f"t = {res_routine.statistic:.2f},  p-värde = {res_routine.pvalue:.4f}")
# p < 0,05: H0 förkastas. Den nya rutinen ger en statistiskt säkerställt kortare handläggningstid.
```

</details>

## Sammanfattning

- En hypotesprövning frågar: **om $H_0$ vore sann, hur ovanlig är då vår data?** Svaret är p-värdet.
- Testvariabeln är *avvikelse / medelfel*: hur många medelfel från $H_0$ vi hamnade.
- p < α → förkasta $H_0$, resultatet är statistiskt säkerställt.
- Att inte förkasta $H_0$ är inte ett bevis för $H_0$.
- p-värdet är inte sannolikheten att $H_0$ är sann.
- Ett 95 % konfidensintervall innehåller precis de värden som inte förkastas på nivån 5 %.

| Fråga | Test | scipy |
|---|---|---|
| Är medelvärdet lika med ett givet värde? | t-test för ett stickprov | `stats.ttest_1samp(x, mu_0)` |
| Skiljer sig två gruppers medelvärden? | t-test för två oberoende stickprov (Welch) | `stats.ttest_ind(a, b, equal_var=False)` |

### Kontrollfrågor
1. Ett test ger p = 0,03. Vilka påståenden är korrekta?
   a) Sannolikheten att $H_0$ är sann är 3 %.
   b) Om $H_0$ är sann skulle ett minst så extremt resultat uppstå i 3 % av fallen.
   c) På signifikansnivån 5 % förkastas $H_0$.
   d) På signifikansnivån 1 % förkastas $H_0$.
2. Ett test ger p = 0,40. Kollegan säger: "Då har vi visat att det inte finns någon skillnad." Vad svarar du?

<details>
<summary>💡 <b>Svar</b></summary>

1. b och c. Påstående a är det vanligaste missförståndet. d är fel eftersom 0,03 > 0,01.
2. Nej. Vi har inte kunnat påvisa någon skillnad, vilket inte är samma sak. Stickprovet kan ha varit för litet.
   Titta på konfidensintervallet: är det brett finns det gott om utrymme för en verklig skillnad.

</details>

**Nästa notebook:** [9. Jämföra två andelar (A/B-test)](../09_tva_andelar/09_tva_andelar.ipynb)